In [1]:
import pandas as pd

import pandas as pd
import ast

# Load your dataset
df = pd.read_csv("Data Inggris - Character_Extract.csv")

# asumsi df sudah ada
df['character'] = df['character'].astype(str).str.strip()

def normalize_alias_custom(name):
    name = name.lower().strip()
    tokens = name.split()
    return " ".join(tokens)

# Normalisasi
df['normalized_characters'] = df['character'].apply(normalize_alias_custom)

# Keep relevant columns
df = df[['storyID', 'sentenceID', 'character', 'normalized_characters']]

# Save
df.to_csv("normalized_characters_all_inggris_v2.csv", index=False)
print("All stories processed and saved as normalized_characters_all.csv")


All stories processed and saved as normalized_characters_all.csv


In [2]:
pip install textdistance


In [3]:
import pandas as pd
import textdistance


In [4]:
import re
import ast
import pandas as pd
import textdistance

def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        if x.startswith("[") and x.endswith("]"):
            try:
                return ast.literal_eval(x)
            except Exception:
                return [x]
        return [x]
    return []

# =========================
# Load data
# =========================
df = pd.read_csv("normalized_characters_all_inggris_v2.csv")
df["normalized_characters"] = df["normalized_characters"].apply(safe_parse)

# --- Rapikan StoryID bila perlu (ambil angka pertama di string) ---
def normalize_story_id(v):
    if pd.isna(v):
        return None
    s = str(v)
    m = re.search(r"\d+", s)
    return int(m.group(0)) if m else s  # jika tak ada angka, pakai string as-is

if "storyID" not in df.columns:
    raise KeyError("Kolom 'StoryID' tidak ditemukan pada CSV.")
df["storyID"] = df["storyID"].apply(normalize_story_id)

# =========================
# Util
# =========================
def normalize(s: str) -> str:
    return s.lower().strip()

def jw_sim(a: str, b: str) -> float:
    # Jaccard similarity (0..1, makin besar makin mirip)
    return float(textdistance.jaro_winkler(a, b))

# =========================
# Clustering berbasis Jaccard
# =========================
def cluster_aliases(characters_list, jw_threshold=0.85):
    """
    - Tanpa pointer, tanpa exclude pairs, tanpa blacklist
    - Aturan:
      1) normalisasi lower/strip
      2) gabung jika substring (kedua arah)
      3) jika keduanya berakhiran sufiks umum ('ne' dsb.), bandingkan akar kata (tanpa sufiks) pakai Jaccard
      4) hindari false positive utk token <4: hanya gabung jika persis sama
      5) selain itu, pakai Jaccard >= threshold
    """
    clusters = {}   # key: Tokoh-i -> set(alias)
    next_id = 1

    for ch in characters_list:
        ch = normalize(ch)
        placed = False

        for key, cluster in clusters.items():
            for name in list(cluster):
                name_n = normalize(name)

                # 1) Substring match dua arah
                if f" {ch} " in f" {name_n} " or f" {name_n} " in f" {ch} ":
                    cluster.add(ch); placed = True; break

                # 4) Similarity Jaccard umum
                if jw_sim(ch, name_n) >= jw_threshold:
                    cluster.add(ch); placed = True; break

            if placed:
                break

        if not placed:
            clusters[f"Person-{next_id}"] = {ch}
            next_id += 1

    # set -> list, urutkan agar rapi (panjang desc)
    return {k: sorted(list(v), key=len, reverse=True) for k, v in clusters.items()}

# =========================
# Proses per StoryID (tanpa hubungan antar story)
# =========================
THRESH = 0.85  # catatan: Jaccard cenderung ketat; turunkan jika terlalu pecah (mis. 0.85)
all_results = []

# buang baris tanpa storyid atau tanpa karakter
df = df[df["storyID"].notna()]
df = df[df["normalized_characters"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

# Flatten list per StoryID
grouped = df.groupby("storyID")["normalized_characters"].apply(lambda x: sum(x, []))

for story_id, characters in grouped.items():
    uniq_chars = sorted(set(characters), key=len, reverse=True)
    result = cluster_aliases(uniq_chars, jw_threshold=THRESH)

    for person, aliases in result.items():
        all_results.append({
            "storyID": story_id,
            "characterID": person,
            "AliasCharacters": aliases
        })

# =========================
# Save
# =========================
df_result = pd.DataFrame(all_results)
outfile = f"alias_jw_{int(THRESH*100)}_english_v2.csv"  # contoh: alias_jaccard_95.csv
df_result.to_csv(outfile, index=False)
print(f"✅ Alias clustering selesai (tanpa pointer/exclude/blacklist), per-story only, Jw≥{THRESH:.2f}. Disimpan ke {outfile}")


✅ Alias clustering selesai (tanpa pointer/exclude/blacklist), per-story only, Jw≥0.85. Disimpan ke alias_jw_85_english_v2.csv


In [5]:
import pandas as pd
import ast

alias_file = 'alias_jw_85_english.csv'
gt_file = 'Data Inggris - Characters_Alias_Reviewed.csv'

def evaluate_extraction(alias_file, gt_file):
    # Load data
    df_alias = pd.read_csv(alias_file)
    df_gt = pd.read_csv(gt_file)

    # Fungsi untuk membersihkan dan menormalisasi teks
    def clean_list(val):
        try:
            l = ast.literal_eval(val)
            return [str(x).lower().strip() for x in l]
        except:
            return [str(val).lower().strip()]

    def clean_str(val):
        return str(val).lower().strip()

    # Preprocessing
    df_alias['cleaned_aliases'] = df_alias['AliasCharacters'].apply(clean_list)
    df_gt['cleaned_char'] = df_gt['AliasCharacters'].apply(clean_str)

    stories = df_gt['storyID'].unique()
    all_metrics = []
    total_tp, total_fp, total_fn = 0, 0, 0

    for sid in stories:
        # Karakter yang seharusnya ada (Ground Truth)
        gt_chars = set(df_gt[df_gt['storyID'] == sid]['cleaned_char'])

        # Karakter yang diprediksi oleh file alias
        pred_entities = df_alias[df_alias['storyID'] == sid]['cleaned_aliases'].tolist()

        tp = 0
        matched_gt = set()
        matched_pred_idx = set()

        # Logika pencocokan: Jika nama GT ada dalam daftar alias prediksi
        for g_char in gt_chars:
            for i, p_aliases in enumerate(pred_entities):
                if g_char in p_aliases:
                    tp += 1
                    matched_gt.add(g_char)
                    matched_pred_idx.add(i)
                    break

        fp = len(pred_entities) - len(matched_pred_idx)
        fn = len(gt_chars) - len(matched_gt)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        # Hitung metrik per cerita (Macro)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        acc = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0

        all_metrics.append({'Precision': prec, 'Recall': rec, 'F1': f1, 'Accuracy': acc})

    # Kalkulasi Rata-rata
    df_m = pd.DataFrame(all_metrics)

    print("=== Macro Average (per-story mean) ===")
    print(f"Accuracy : {df_m['Accuracy'].mean():.4f}")
    print(f"Precision: {df_m['Precision'].mean():.4f}")
    print(f"Recall   : {df_m['Recall'].mean():.4f}")
    print(f"F1       : {df_m['F1'].mean():.4f}")

    # Micro Average
    micro_prec = total_tp / (total_tp + total_fp)
    micro_rec = total_tp / (total_tp + total_fn)
    micro_f1 = 2 * micro_prec * micro_rec / (micro_prec + micro_rec)
    micro_acc = total_tp / (total_tp + total_fp + total_fn)

    print("\n=== Overall (Micro) ===")
    print(f"Accuracy : {micro_acc:.4f}")
    print(f"Precision: {micro_prec:.4f}")
    print(f"Recall   : {micro_rec:.4f}")
    print(f"F1       : {micro_f1:.4f}")

# Jalankan evaluasi
evaluate_extraction('alias_jw_85_english_v2.csv', 'Data Inggris - Characters_Alias_Reviewed.csv')

=== Macro Average (per-story mean) ===
Accuracy : 0.8723
Precision: 0.8923
Recall   : 0.9629
F1       : 0.9235

=== Overall (Micro) ===
Accuracy : 0.8571
Precision: 0.8880
Recall   : 0.9611
F1       : 0.9231


In [7]:
import pandas as pd
import ast
from collections import defaultdict
import re

# 1. Load data
df = pd.read_csv("alias_jw_85_english_v2.csv")

def parse_aliases(val):
    try:
        return ast.literal_eval(val)
    except:
        return [str(val).strip()]

df["aliases"] = df["AliasCharacters"].apply(parse_aliases)

# 2. Daftar Peran Bahasa Inggris (WSM)
ROLE_KEYWORDS_ENGLISH = [
    # Keluarga
    ('father', 'father'), ('dad', 'father'), ('papa', 'father'), ('pa', 'father'),
    ('mother', 'mother'), ('mom', 'mother'), ('mama', 'mother'), ('ma', 'mother'),
    ('son', 'child'), ('daughter', 'child'), ('child', 'child'), ('children', 'child'),
    ('brother', 'sibling'), ('sister', 'sibling'),
    ('husband', 'spouse'), ('wife', 'spouse'),
    ('grandfather', 'grandparent'), ('grandmother', 'grandparent'),

    # Gelar / Kerajaan
    ('king', 'monarch'), ('queen', 'monarch'), ('emperor', 'monarch'), ('empress', 'monarch'),
    ('prince', 'royalty'), ('princess', 'royalty'),
    ('lord', 'noble'), ('lady', 'noble'), ('sir', 'noble'),
    ('minister', 'official'), ('preacher', 'official'), ('priest', 'official'),

    # Gender/Usia
    ('man', 'man'), ('men', 'man'),
    ('woman', 'woman'), ('women', 'woman'),
    ('boy', 'boy'), ('boys', 'boy'),
    ('girl', 'girl'), ('girls', 'girl'),

    # Peran Umum dalam Cerita
    ('thief', 'criminal'), ('robber', 'criminal'),
    ('servant', 'servant'), ('maid', 'servant'),
    ('friend', 'friend'),
]

# 3. Pola Ordinal Bahasa Inggris
ORDINAL_PATTERN = re.compile(
    r'\b(?:first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth|1st|2nd|3rd|[0-9]+th)\b',
    re.IGNORECASE
)

def normalize(text):
    return text.lower().strip()

def get_role(alias):
    alias = normalize(alias)
    words = alias.split()
    for keyword, role in ROLE_KEYWORDS_ENGLISH:
        if alias == keyword or keyword in words:
            return role
    return None

def contains_ordinal(alias):
    return bool(ORDINAL_PATTERN.search(normalize(alias)))

# 4. Pengelompokan (Clustering)
grouped = defaultdict(list)
for _, row in df.iterrows():
    grouped[row['storyID']].append({
        'person': row['characterID'],
        'aliases': [normalize(a) for a in row['aliases']]
    })

final_results = []
for story_id, clusters in grouped.items():
    merged = []
    visited = [False] * len(clusters)

    for i in range(len(clusters)):
        if visited[i]: continue

        current_aliases = set(clusters[i]['aliases'])
        merged_indices = [i]
        roles_i = {get_role(a) for a in current_aliases if get_role(a)}

        for j in range(i + 1, len(clusters)):
            if visited[j]: continue

            other_aliases = set(clusters[j]['aliases'])
            roles_j = {get_role(a) for a in other_aliases if get_role(a)}

            # Jangan gabungkan jika ada penanda urutan (ordinal)
            if any(contains_ordinal(a) for a in current_aliases | other_aliases):
                continue

            # Gabungkan jika peran sama
            if roles_i and roles_i == roles_j:
                current_aliases |= other_aliases
                visited[j] = True
                merged_indices.append(j)

        merged.append({
            "aliases": list(current_aliases),
            "role": next(iter(roles_i)) if roles_i else None
        })
        for idx in merged_indices:
            visited[idx] = True

    for idx, group in enumerate(merged, 1):
        final_results.append({
            "storyID": story_id,
            "characterID": f"Tokoh-{idx}",
            "AliasCharacters": sorted(set(group['aliases']))
        })

# 5. Simpan Hasil
df_out = pd.DataFrame(final_results)
df_out.to_csv("alias_jw_85_english_wsm_v2.csv", index=False)
print("✅ Selesai! File disimpan sebagai alias_jw_85_english_wsm_v2.csv")

✅ Selesai! File disimpan sebagai alias_jw_85_english_wsm_v2.csv


In [8]:
import pandas as pd
import ast

alias_file = 'alias_jw_85_english_wsm_v2.csv'
gt_file = 'Data Inggris - Characters_Alias_Reviewed.csv'

def evaluate_extraction(alias_file, gt_file):
    # Load data
    df_alias = pd.read_csv(alias_file)
    df_gt = pd.read_csv(gt_file)

    # Fungsi untuk membersihkan dan menormalisasi teks
    def clean_list(val):
        try:
            l = ast.literal_eval(val)
            return [str(x).lower().strip() for x in l]
        except:
            return [str(val).lower().strip()]

    def clean_str(val):
        return str(val).lower().strip()

    # Preprocessing
    df_alias['cleaned_aliases'] = df_alias['AliasCharacters'].apply(clean_list)
    df_gt['cleaned_char'] = df_gt['AliasCharacters'].apply(clean_str)

    stories = df_gt['storyID'].unique()
    all_metrics = []
    total_tp, total_fp, total_fn = 0, 0, 0

    for sid in stories:
        # Karakter yang seharusnya ada (Ground Truth)
        gt_chars = set(df_gt[df_gt['storyID'] == sid]['cleaned_char'])

        # Karakter yang diprediksi oleh file alias
        pred_entities = df_alias[df_alias['storyID'] == sid]['cleaned_aliases'].tolist()

        tp = 0
        matched_gt = set()
        matched_pred_idx = set()

        # Logika pencocokan: Jika nama GT ada dalam daftar alias prediksi
        for g_char in gt_chars:
            for i, p_aliases in enumerate(pred_entities):
                if g_char in p_aliases:
                    tp += 1
                    matched_gt.add(g_char)
                    matched_pred_idx.add(i)
                    break

        fp = len(pred_entities) - len(matched_pred_idx)
        fn = len(gt_chars) - len(matched_gt)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        # Hitung metrik per cerita (Macro)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        acc = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0

        all_metrics.append({'Precision': prec, 'Recall': rec, 'F1': f1, 'Accuracy': acc})

    # Kalkulasi Rata-rata
    df_m = pd.DataFrame(all_metrics)

    print("=== Macro Average (per-story mean) ===")
    print(f"Accuracy : {df_m['Accuracy'].mean():.4f}")
    print(f"Precision: {df_m['Precision'].mean():.4f}")
    print(f"Recall   : {df_m['Recall'].mean():.4f}")
    print(f"F1       : {df_m['F1'].mean():.4f}")

    # Micro Average
    micro_prec = total_tp / (total_tp + total_fp)
    micro_rec = total_tp / (total_tp + total_fn)
    micro_f1 = 2 * micro_prec * micro_rec / (micro_prec + micro_rec)
    micro_acc = total_tp / (total_tp + total_fp + total_fn)

    print("\n=== Overall (Micro) ===")
    print(f"Accuracy : {micro_acc:.4f}")
    print(f"Precision: {micro_prec:.4f}")
    print(f"Recall   : {micro_rec:.4f}")
    print(f"F1       : {micro_f1:.4f}")

# Jalankan evaluasi
evaluate_extraction('alias_jw_85_english_wsm_v2.csv', 'Data Inggris - Characters_Alias_Reviewed.csv')

=== Macro Average (per-story mean) ===
Accuracy : 0.9247
Precision: 0.9475
Recall   : 0.9629
F1       : 0.9542

=== Overall (Micro) ===
Accuracy : 0.9037
Precision: 0.9380
Recall   : 0.9611
F1       : 0.9494
